In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import joblib

# Load processed data
df = pd.read_csv('../data/processed/hotel_bookings_processed.csv')

# Define feature columns (must match what was created in preprocessing)
feature_cols = [
    'lead_time', 'total_nights', 'total_guests', 'is_weekend', 'is_peak_season',
    'day_of_week', 'arrival_date_month_num', 'is_repeated_guest', 'previous_cancellations',
    'booking_changes', 'required_car_parking_spaces', 'total_of_special_requests',
    'hotel_encoded', 'meal_encoded', 'market_segment_encoded', 'distribution_channel_encoded',
    'reserved_room_type_encoded', 'deposit_type_encoded', 'customer_type_encoded'
]

# Prepare features and target
X = df[feature_cols]
y = df['adr']  # Price is our target

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")

# Train model
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Evaluate
y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"\nModel Performance:")
print(f"MSE: {mse:.2f}")
print(f"R²: {r2:.3f}")
print(f"RMSE: ${np.sqrt(mse):.2f}")

# Feature importance
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print("\nTop 10 Most Important Features:")
print(feature_importance.head(10))

# Save model and feature columns
joblib.dump(model, '../src/models/demand_model.pkl')
joblib.dump(feature_cols, '../src/models/feature_columns.pkl')  # Save feature columns too
print("✅ Model and feature columns saved!")

# Simple price optimization
def optimize_price(features_df, model, price_range=(50, 300), steps=20):
    """Find optimal price for given features"""
    prices = np.linspace(price_range[0], price_range[1], steps)
    revenues = []
    
    for price in prices:
        # Predict demand at this price (simplified)
        predicted_demand = model.predict(features_df)[0]
        # Simple demand curve: higher price = lower demand
        adjusted_demand = max(0, predicted_demand * (1 - (price - 100) / 200))
        revenue = price * adjusted_demand
        revenues.append(revenue)
    
    optimal_idx = np.argmax(revenues)
    return prices[optimal_idx], revenues[optimal_idx]

# Example usage - use DataFrame instead of array to avoid warnings
if len(X_test) > 0:
    sample_features_df = X_test.iloc[[0]]  # Keep as DataFrame
    optimal_price, max_revenue = optimize_price(sample_features_df, model)
    print(f"\nPrice Optimization Example:")
    print(f"Optimal Price: ${optimal_price:.2f}")
    print(f"Expected Revenue: ${max_revenue:.2f}")

print("\n✅ Model training complete!")


Features shape: (73419, 19)
Target shape: (73419,)
Training set: (58735, 19)
Test set: (14684, 19)

Model Performance:
MSE: 421.84
R²: 0.818
RMSE: $20.54

Top 10 Most Important Features:
                       feature  importance
2                 total_guests    0.184644
6       arrival_date_month_num    0.137077
16  reserved_room_type_encoded    0.125062
12               hotel_encoded    0.118101
0                    lead_time    0.107971
4               is_peak_season    0.087850
14      market_segment_encoded    0.053980
13                meal_encoded    0.042474
1                 total_nights    0.034604
5                  day_of_week    0.031407
✅ Model and feature columns saved!

Price Optimization Example:
Optimal Price: $155.26
Expected Revenue: $10422.23

✅ Model training complete!
